# Chapter 23 — What Should a Translation Preserve?

**Book alignment:** *Embeddings From First Principles*, Chapter 23

**Notebook role:** `SYNTHETIC_DEMO + ARTIFACT_REPLAY` — sections 1-6 construct a controlled toy example of the mechanism (reproducing no benchmark); the final section replays the frozen Wave 6 artifact and re-derives the numbers the chapter quotes.

**Question this notebook isolates:** If source and target encoders organize the same objects
differently, what happens when a translation objective is told to preserve the **source**
geometry anyway?

This is a deterministic **toy diagnostic** for the chapter's reader lab. It does **not** reproduce
the measured mxbai→Qwen3 source-VSP weight sweep in the chapter, and it does not establish that
target-rank distillation solves real cross-space translation.

In [1]:
import numpy as np

rng = np.random.default_rng(23)

N = 500
D = 6
K = 10
N_CLUSTERS = 10
NOISE = 0.35


def normalize_rows(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)


def cosine_matrix(x, y):
    return normalize_rows(x) @ normalize_rows(y).T


def topk_without_self(scores, i, k=K):
    return [j for j in np.argsort(-scores) if j != i][:k]

## 1. Construct source and target spaces that genuinely disagree

The source space `A` contains clustered objects.

The target space `B` is produced by an orthogonal rotation **plus strong anisotropic scaling**.
The scaling is deliberate: it changes angles and neighborhoods, so `G_A` and `G_B` are not the
same geometry in different coordinates.

In [2]:
centers = rng.normal(size=(N_CLUSTERS, D))
centers = normalize_rows(centers)

labels = np.arange(N) % N_CLUSTERS
A = np.vstack([
    centers[label] + NOISE * rng.normal(size=D)
    for label in labels
])
A = normalize_rows(A)

R, _ = np.linalg.qr(rng.normal(size=(D, D)))

scales = np.ones(D)
scales[:2] = 8.0
scales[2:4] = 1.0 / 8.0
scales[4:] = 2.0

B = normalize_rows(A @ R @ np.diag(scales))

S_A = cosine_matrix(A, A)
S_B = cosine_matrix(B, B)

N_A = [topk_without_self(S_A[i], i, K) for i in range(N)]
N_B = [topk_without_self(S_B[i], i, K) for i in range(N)]

native_agreement = np.mean([
    len(set(N_A[i]) & set(N_B[i])) / K
    for i in range(N)
])

print(f"native A-vs-B agreement@{K}: {native_agreement:.3f}")
print("The spaces disagree enough that source-isometry and target fidelity can conflict.")

assert native_agreement < 0.60

native A-vs-B agreement@10: 0.299
The spaces disagree enough that source-isometry and target fidelity can conflict.


## 2. Use one map family and vary only the preservation objective

To isolate the objective, every candidate comes from the same one-parameter family:

`T_lambda(A) = normalize(A @ R @ diag(scales ** lambda))`

- `lambda = 0` applies only the orthogonal rotation and therefore preserves source cosine geometry.
- `lambda = 1` reproduces the target-generating anisotropy and therefore reproduces target geometry.
- values between them trade source-isometry against target behavior.

This is intentionally simpler than training an MLP: it removes optimizer and capacity confounds.

In [3]:
pair_ix = np.triu_indices(N, k=1)
source_pairwise = S_A[pair_ix]
target_pairwise = S_B[pair_ix]


def translate(lam):
    return normalize_rows(A @ R @ np.diag(scales ** lam))


def evaluate(T):
    T = normalize_rows(T)
    S_TB = cosine_matrix(T, B)

    ranking = np.argsort(-S_TB, axis=1)
    paired_rank = np.array([
        np.where(ranking[i] == i)[0][0]
        for i in range(N)
    ])

    translated_neighbors = [
        topk_without_self(S_TB[i], i, K)
        for i in range(N)
    ]

    agreement = np.mean([
        len(set(N_B[i]) & set(translated_neighbors[i])) / K
        for i in range(N)
    ])

    return {
        "cosine_to_target": float(np.mean(np.sum(T * B, axis=1))),
        "paired_recall@10": float(np.mean(paired_rank < 10)),
        "mrr": float(np.mean(1.0 / (paired_rank + 1))),
        "agreement@10": float(agreement),
    }


def losses(T):
    T = normalize_rows(T)
    S_T = cosine_matrix(T, T)

    point_loss = 1.0 - np.mean(np.sum(T * B, axis=1))
    source_geometry_loss = np.mean((S_T[pair_ix] - source_pairwise) ** 2)
    target_geometry_loss = np.mean((S_T[pair_ix] - target_pairwise) ** 2)

    return {
        "point": float(point_loss),
        "source_geometry": float(source_geometry_loss),
        "target_geometry": float(target_geometry_loss),
    }

## 3. Sweep the same map family

We will select three translators from the same candidates:

- **A — point reconstruction:** minimize point loss.
- **B — point + SOURCE preservation:** minimize point loss plus a source-geometry penalty.
- **C — point + TARGET geometry:** minimize point loss plus a target-geometry penalty.

The third objective is a **toy target-geometry surrogate**, not the chapter's proposed
differentiable target-rank distillation loss.

In [4]:
lambdas = np.linspace(0.0, 1.2, 61)
records = []

for lam in lambdas:
    T = translate(lam)
    rec = {"lambda": float(lam), **losses(T), **evaluate(T)}
    records.append(rec)

SOURCE_WEIGHT = 20.0
TARGET_WEIGHT = 20.0

point_choice = min(records, key=lambda r: r["point"])
source_choice = min(
    records,
    key=lambda r: r["point"] + SOURCE_WEIGHT * r["source_geometry"],
)
target_choice = min(
    records,
    key=lambda r: r["point"] + TARGET_WEIGHT * r["target_geometry"],
)

choices = {
    "A point only": point_choice,
    "B point + SOURCE geometry": source_choice,
    "C point + TARGET geometry": target_choice,
}

for name, r in choices.items():
    print(
        f"{name:27} lambda={r['lambda']:.2f}  "
        f"cos={r['cosine_to_target']:.3f}  "
        f"R@10={r['paired_recall@10']:.3f}  "
        f"MRR={r['mrr']:.3f}  "
        f"agree@10={r['agreement@10']:.3f}"
    )

A point only                lambda=1.00  cos=1.000  R@10=1.000  MRR=1.000  agree@10=1.000
B point + SOURCE geometry   lambda=0.02  cos=0.734  R@10=0.932  MRR=0.329  agree@10=0.385
C point + TARGET geometry   lambda=1.00  cos=1.000  R@10=1.000  MRR=1.000  agree@10=1.000


## 4. The causal comparison

The source-preserving objective should pull `lambda` back toward zero — toward the source shape —
even though the destination consumer lives in target space.

That is the category error the chapter is warning about.

In [5]:
assert point_choice["lambda"] > 0.90
assert target_choice["lambda"] > 0.90
assert source_choice["lambda"] < 0.20

assert source_choice["cosine_to_target"] < point_choice["cosine_to_target"]
assert source_choice["agreement@10"] < point_choice["agreement@10"]
assert source_choice["paired_recall@10"] <= point_choice["paired_recall@10"]

print("source-preservation intervention:")
print(f"  lambda:        {point_choice['lambda']:.2f} -> {source_choice['lambda']:.2f}")
print(f"  target cosine: {point_choice['cosine_to_target']:.3f} -> {source_choice['cosine_to_target']:.3f}")
print(f"  paired R@10:   {point_choice['paired_recall@10']:.3f} -> {source_choice['paired_recall@10']:.3f}")
print(f"  agreement@10:  {point_choice['agreement@10']:.3f} -> {source_choice['agreement@10']:.3f}")

print("\nThe source-preserving loss is doing exactly what it was asked to do.")
print("The problem is that source geometry is the wrong authority for this consumer.")

source-preservation intervention:
  lambda:        1.00 -> 0.02
  target cosine: 1.000 -> 0.734
  paired R@10:   1.000 -> 0.932
  agreement@10:  1.000 -> 0.385

The source-preserving loss is doing exactly what it was asked to do.
The problem is that source geometry is the wrong authority for this consumer.


## 5. Show that the source objective really did preserve source geometry

A bad result is not enough. We need to show the intervention achieved its intended invariant.

If source pairwise cosine error falls while target behavior worsens, then the problem is not
"the regularizer failed." The regularizer succeeded at preserving the wrong thing.

In [6]:
point_losses = losses(translate(point_choice["lambda"]))
source_losses = losses(translate(source_choice["lambda"]))

print("pairwise source-geometry loss")
print(f"  point-only translator:     {point_losses['source_geometry']:.6f}")
print(f"  source-preserving choice:  {source_losses['source_geometry']:.6f}")

print("\npairwise target-geometry loss")
print(f"  point-only translator:     {point_losses['target_geometry']:.6f}")
print(f"  source-preserving choice:  {source_losses['target_geometry']:.6f}")

assert source_losses["source_geometry"] < point_losses["source_geometry"]
assert source_losses["target_geometry"] > point_losses["target_geometry"]

print("\nThe source regularizer preserved source structure more faithfully")
print("while moving the translated representation away from target structure.")

pairwise source-geometry loss
  point-only translator:     0.248486
  source-preserving choice:  0.000510

pairwise target-geometry loss
  point-only translator:     0.000000
  source-preserving choice:  0.231339

The source regularizer preserved source structure more faithfully
while moving the translated representation away from target structure.


## 6. Modify the preservation authority

Try changing `SOURCE_WEIGHT`, the anisotropic `scales`, or the source/target disagreement.

A useful exercise is to sweep `SOURCE_WEIGHT` and observe the Pareto curve between source-isometry
and target fidelity.

In [7]:
print(f"{'source weight':>13} {'lambda':>8} {'target cos':>11} {'R@10':>8} {'agree@10':>10}")

for weight in (0.0, 0.5, 1.0, 3.0, 10.0, 20.0, 50.0):
    choice = min(
        records,
        key=lambda r: r["point"] + weight * r["source_geometry"],
    )
    print(
        f"{weight:13.1f} {choice['lambda']:8.2f} "
        f"{choice['cosine_to_target']:11.3f} "
        f"{choice['paired_recall@10']:8.3f} "
        f"{choice['agreement@10']:10.3f}"
    )

source weight   lambda  target cos     R@10   agree@10
          0.0     1.00       1.000    1.000      1.000
          0.5     0.58       0.979    1.000      0.742
          1.0     0.34       0.926    1.000      0.578
          3.0     0.12       0.813    0.974      0.443
         10.0     0.04       0.751    0.940      0.395
         20.0     0.02       0.734    0.932      0.385
         50.0     0.00       0.717    0.930      0.374


## What we earned

This controlled experiment makes the chapter's central rule explicit:

- **Source-isometry**, **point reconstruction**, and **target-geometry fidelity** are different
  objectives.
- When `G_A != G_B`, preserving source geometry can actively fight target behavior.
- A preservation loss can therefore **succeed at its own metric and still make the bridge worse
  for the destination consumer**.
- The correct preservation authority comes from the contract:
  - source geometry for an isometric migration,
  - target geometry when translated vectors must behave like native target vectors,
  - downstream task labels when neither geometry is the real objective.
- Capacity and optimization are deliberately controlled away here; only the chosen invariant
  changes.

The manuscript contains a separate **measured** mxbai→Qwen3 diagnostic where adding source
cosine-VSP sharply degraded target recovery and neighborhood agreement. This notebook is a toy
causal model of the same design principle, not a reproduction of those measured numbers.

**Next:** Chapter 24 applies the preservation discipline to compression: can a compressed document
retain coarse embedding geometry while silently losing or reversing a claim?

## 6. Replay the measured Wave 6 source-VSP sweep

The toy above shows the mechanism. This section reads the frozen Wave 6 VSP sweep
(`experiments/embeddings-from-first-principles/wave6/artifacts/vsp-sweep.json`) and checks that
raising the source cosine-preservation weight degrades target fidelity monotonically.


In [ ]:
from pathlib import Path
import json


def _vsp():
    for c in (Path.cwd(), *Path.cwd().parents):
        p = c / 'experiments/embeddings-from-first-principles/wave6/artifacts/vsp-sweep.json'
        if p.exists():
            return json.loads(p.read_text(encoding='utf-8'))
    raise RuntimeError('run from a checkout with the wave6 artifact')


sweep = sorted(_vsp(), key=lambda r: r['vsp'])
print(f"{'VSP weight':>10}  {'cosine':>7}  {'R@1':>6}  {'agree@10':>9}  {'order':>6}")
for r in sweep:
    print(f"{r['vsp']:>10.1f}  {r['cosine_to_target']:>7.3f}  {r['recall_at_1']:>6.3f}  "
          f"{r['agreement_at_10']:>9.3f}  {r['order_preservation_local']:>6.2f}")

cos = [r['cosine_to_target'] for r in sweep]
r1 = [r['recall_at_1'] for r in sweep]
ag = [r['agreement_at_10'] for r in sweep]
for seq, label in [(cos, 'cosine'), (r1, 'Recall@1'), (ag, 'agreement@10')]:
    assert all(seq[i] >= seq[i + 1] - 0.01 for i in range(len(seq) - 1)), label
assert cos[0] - cos[1] < 0.02      # a small term is nearly inert
assert r1[0] - r1[-1] > 0.2        # a large term costs > 20 points of Recall@1
print('source-VSP trades target fidelity for source fidelity, monotonically in its weight')
